# Create a virtual environment & load necessary packages.


In [ ]:
!pip install virtualenv
!virtualenv bip
!source bip/bin/activate
!pip install z3-solver
!pip install onnx
!git clone https://github.com/ytsao/lattice-gym

In [ ]:
from z3 import *
from z3.z3util import get_vars

# Tutorial 1: What is Z3?
Z3 is a satisfied modulo theory (SMT) solver developed by Microsoft.

SMT solver is more general than satisfied (SAT) solver.
It can solve general problems not only restricted to Boolean problems.


For more details, you can check:
* Wikipedia: https://en.wikipedia.org/wiki/Z3_Theorem_Prover
* Source code: https://github.com/z3prover/z3
* Official website: https://microsoft.github.io/z3guide/docs/logic/intro/

In [ ]:
# Define a solver
solver = Solver()

# Declare variables
# Boolean variables
# not b1 /\ b1 /\ b2
b1 = Bool("b1")
b2 = Bool("b2")
solver.add(Not(b1))
solver.add(And(b1, b2))
status = solver.check()
print("Status:", status)
if status == sat:
    print(solver.model())

solver.reset()

In [ ]:
# Integer variables
# 0 <= i1 <= 10 /\ 0 <= i2 <= 10 /\ 0 <= i3 <= 10 /\ i1 + i2 + i3 == 10
i1 = Int("i1")
i2 = Int("i2")
i3 = Int("i3")
solver.add(And(i1 >= 0, i1 <= 10))
solver.add(And(i2 >= 0, i2 <= 10))
solver.add(And(i3 >= 0, i3 <= 10))
solver.add(i1 + i2 + i3 == 10)
status = solver.check()
print("Status:", status)
if status == sat:
    print(solver.model())

solver.reset()

In [ ]:
# Real variables
# 0 <= r1 <= 10 /\ 0 <= r2 <= 10 /\ r1 + r2 == 10
r1 = Real("r1")
r2 = Real("r2")
solver.add(And(r1 >= 0, r1 <= 10))
solver.add(And(r2 >= 0, r2 <= 10))
solver.add(r1 + r2 == 10)
status = solver.check()
print("Status:", status)
if status == sat:
    print(solver.model())

# Tutorial 2: How to verify neural networks by using z3?

![alt text](./images/nnv.png "nnv example")


**preconditions:**

$1 \leq x_1 \land x_1 \leq 2$

$2 \leq x_2 \land x_2 \leq 3$

**hidden layer:**

$h_1 = 0.8 * x_1 - 0.7 * x_2$

$h_2 = 0.6 * x_1 + 0.5 * x_2$

**Relu layer:**

$a_1 = ReLU(h_1, 0.0) \iff max(h_1, 0.0)$

$a_1 = ReLU(h_2, 0.0) \iff max(h_2, 0.0)$

But, there is no `max` function in Z3 Python API can be used.

How to deal with it?

**output layer & postcondition:**

$o_1 = -1.0 * a_1 + 0.4 * a_2$

$o_1 < 0.5$

In [ ]:
def verify():
  # TODO: 

  return

verify()

# Exercise 1:

In the previous practice, our code works!
However, it is not flexible to extend to other instances.

For example, what if we have more layers in the network? or we have different input/output dimensions? 

In the following exercises, we are going to use a general way to load a neural network, preconditions, and postconditions such that we can directly build a model.

In general, we have two input files with `onnx` and `vnnlib` format, respectively.
`onnx` file defines a neural network architecture and stores all weights and biases. 
It gives a way to store different networks from different libraries. 

`vnnlib` file is a subset of `SMT` file. It means that `SMT` parser logic can be applied to `vnnlib` file since the syntax is the same.
Hence, we can use `z3` solver to load the preconditions and postconditions. 

For more information about `onnx` and `vnnlib`, you refer the following links:
1. https://onnx.ai/
2. https://en.wikipedia.org/wiki/Open_Neural_Network_Exchange
3. https://www.vnnlib.org/


We provide different instances in the following table:
| ONNX Model File | VNNLIB Specification File |
| :--- | :--- |
| `instances/onnx/ToyNet.onnx` |     `instances/vnnlib/ToyNet.vnnlib` |
| `instances/onnx/test_nano.onnx` |  `instances/vnnlib/test_nano.vnnlib` |
| `instances/onnx/test_tiny.onnx` |  `instances/vnnlib/test_tiny.vnnlib` |
| `instances/onnx/test_small.onnx` | `instances/vnnlib/test_small.vnnlib` |
| `instances/onnx/sat_v6_c27.onnx` | `instances/vnnlib/sat_v6_c27.vnnlib` |

In [ ]:
import os
import math
import onnx
from onnx import numpy_helper

def load_network(filename: str):
  if os.path.exists(filename):
    return onnx.load(filename)
  raise FileNotFoundError(f"File {filename} not found.")

def load_vnnlib(filename: str):
  if os.path.exists(filename):
    constraints = parse_smt2_file(filename)
    return constraints
  raise FileNotFoundError(f"File {filename} not found.")

class Layer:
  def __init__(self):
    self.type = None
    self.size = None
    self.neurons = []
    self.weights = None
    self.biases = None

def build(network):
  mp_init = {}
  for init in network.graph.initializer:
    mp_init[init.name] = init

  layers = []

  # Input layer: derive size from the declared input shape (general: product of all non-batch dims)
  graph_input = network.graph.input[0]
  dims = [d.dim_value if d.dim_value > 0 else 1
          for d in graph_input.type.tensor_type.shape.dim]
  # skip batch dimension (index 0), multiply the rest
  input_dimensions = math.prod(dims[1:]) if len(dims) > 1 else dims[0]
  input_layer = Layer()
  input_layer.type = "input_layer"
  input_layer.size = input_dimensions
  print(f"Input dimensions: {input_layer.size}")
  for n in range(input_layer.size):
    input_layer.neurons.append(Real(f"X_{n}"))
  layers.append(input_layer)

  previous_layer = input_layer
  for node in network.graph.node:
    layer = Layer()
    layer.type = node.op_type
    for i in node.input:
      if i in mp_init.keys():
        tensor = numpy_helper.to_array(mp_init[i])
        if tensor.ndim == 2:
          layer.weights = tensor
        elif tensor.ndim == 1:
          layer.biases = tensor

    # decide layer size
    if layer.biases is not None:
      layer.size = layer.biases.shape[0]
    elif layer.weights is not None:
      layer.size = layer.weights.shape[0]
    else:
      layer.size = previous_layer.size

    # create variables
    if network.graph.output[0].name == node.output[0]:
      for n in range(layer.size):
        layer.neurons.append(Real(f"Y_{n}"))
    else:
      for n in range(layer.size):
        layer.neurons.append(Real(f"H_{len(layers)}_{n}"))

    layers.append(layer)
    previous_layer = layer
  return layers

In [ ]:
def exercise2(network, properties):
  verifier = Solver()
  layers = build(network)

  for p in properties:
    print(p)
    # TODO: add the preconditions and postconditions

  for id, layer in enumerate(layers):
    if id == 0:  # skip input layer
      continue

    print(f"Layer type: {layer.type}")
    print(f"Size: {layer.size}")
    print(f"Weights: {layer.weights}")
    print(f"Biases: {layer.biases}\n")

    if layer.type == "MatMul":
      # TODO:
      print(f"""MatMul layer processing not implemented yet.
Weights shape: {layer.weights.shape}
Weights: {layer.weights}
""")
    elif layer.type == "Gemm":
      # TODO:
      print(f"""MatMul layer processing not implemented yet.
Weights shape: {layer.weights.shape}
Weights: {layer.weights}
Biases shape: {layer.biases.shape}
Biases: {layer.biases}
""")
    elif layer.type == "Relu":
      # TODO:
      print("""Relu layer processing not implemented yet.
There is no weights or biases for Relu layer.""")
    elif layer.type == "Add":
      # TODO:
      print(f"""Add layer processing not implemented yet.
Biases shape: {layer.biases.shape}
Biases: {layer.biases}
""")
    else:
      raise NotImplementedError(f"Layer type {layer.type} not implemented yet.")

  status = verifier.check()
  print(status)
  return

# network = load_network("/content/lattice-gym/bip/instances/onnx/ToyNet.onnx")
# properties = load_vnnlib("/content/lattice-gym/bip/instances/vnnlib/ToyNet.vnnlib")
network = load_network("./instances/onnx/ToyNet.onnx")
properties = load_vnnlib("./instances/vnnlib/ToyNet.vnnlib")
exercise2(network, properties)

# Exercise 2: Apply forward bound propagation with interval abstraction to verify neural networks.

As we have seen in the morning, we can apply a general forward bound propagation to verify neural networks.

By doing so, we can verify neural networks without invoking any external solvers.

In [ ]:
class Interval:
  def __init__(self, lower, upper):
    self.lower = lower
    self.upper = upper

def interval_prop(network, properties):
  layers = build(network)

  for id, layer in enumerate(layers):

    print(f"Layer type: {layer.type}")
    print(f"Size: {layer.size}")
    print(f"Weights: {layer.weights}")
    print(f"Biases: {layer.biases}")

    if layer.type == "input_layer":
      # TODO: create preconditions
      print()
    elif layer.type == "MatMul":
      # TODO:
      print(f"""MatMul layer processing not implemented yet.
Weights shape: {layer.weights.shape}
Weights: {layer.weights}
""")
    elif layer.type == "Gemm":
      # TODO:
      print(f"""Gemm layer processing not implemented yet.
Weights shape: {layer.weights.shape}
Weights: {layer.weights}
Biases shape: {layer.biases.shape}
Biases: {layer.biases}
""")
    elif layer.type == "Relu":
      # TODO:
      print("""Relu layer processing not implemented yet.
There is no weights or biases for Relu layer.""")
    elif layer.type == "Add":
      # TODO:
      print(f"""Add layer processing not implemented yet.
Biases shape: {layer.biases.shape}
Biases: {layer.biases}
""")
    else:
      raise NotImplementedError(f"Layer type {layer.type} not implemented yet.")

  # TODO: Chcek postconditions

  return

# network = load_network("/content/lattice-gym/bip/instances/onnx/ToyNet.onnx")
# properties = load_vnnlib("/content/lattice-gym/bip/instances/vnnlib/ToyNet.vnnlib")
network = load_network("./instances/onnx/ToyNet.onnx")
properties = load_vnnlib("./instances/vnnlib/ToyNet.vnnlib")
interval_prop(network, properties)